# IPL Scouting Results Notebook

This notebook is the main scouting workflow for the project. It keeps the project focused on scouting rather than forcing a noisy supervised prediction task.

The notebook does four things:

1. Loads the processed SMA/IPL player-season features and EDA summaries.
2. Builds observed role-specific impact scores for SMA and IPL players.
3. Uses K-means clustering to group players into performance profiles.
4. Finds similar IPL players for domestic SMA candidates and creates final scouting shortlists.


In [ ]:
from pathlib import Path
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

PROJECT_ROOT = Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
EDA_DIR = REPORTS_DIR / "eda"
SCOUTING_DIR = REPORTS_DIR / "scouting"
SCOUTING_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_PATH = PROCESSED_DIR / "player_features.parquet"
assert FEATURES_PATH.exists(), f"Missing {FEATURES_PATH}. Run run_features.py first."

features = pd.read_parquet(FEATURES_PATH)
print(f"Loaded player_features: {features.shape[0]:,} rows x {features.shape[1]:,} columns")
print(features[["competition", "role", "season", "player_key", "player_name"]].head())


## 1. Simple EDA Recap

The full EDA lives in `reports/eda`, but the project only needs a simple story:

- How much data do we have?
- How many players overlap between SMA and IPL?
- How different are SMA and IPL environments?
- Why is supervised prediction hard?


In [ ]:
def read_optional_csv(path):
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

eda_overview = read_optional_csv(EDA_DIR / "dataset_overview.csv")
player_overlap = read_optional_csv(EDA_DIR / "player_overlap.csv")
metric_summary = read_optional_csv(EDA_DIR / "metric_summary.csv")
training_summary = read_optional_csv(EDA_DIR / "training_summary.csv")

if not eda_overview.empty:
    print("Dataset coverage:")
    display_cols = ["competition", "role", "player_season_rows", "players", "seasons", "first_season", "last_season"]
    print(eda_overview[display_cols].to_string(index=False))
else:
    print(features.groupby(["competition", "role"]).agg(rows=("player_key", "size"), players=("player_key", "nunique")))

if not player_overlap.empty:
    print("\nPlayer overlap between SMA and IPL:")
    print(player_overlap.to_string(index=False))

if not training_summary.empty:
    print("\nSupervised training subset:")
    print(training_summary[["role", "training_rows", "players", "mean_target_ipl_sample"]].to_string(index=False))

print("\nInterpretation: SMA has a large scouting pool, but only a smaller subset has both prior SMA data and later IPL outcomes. That makes direct supervised prediction noisy.")


## 2. Build Player Profiles

The dashboard should compare players at profile level, not individual innings. For each player, competition, and role, we aggregate all available seasons into one scouting profile.

Minimum sample rules:

- SMA batters/bowlers: at least 15 role innings.
- IPL batters/bowlers: at least 10 role innings.

These thresholds reduce tiny-sample outliers.


In [ ]:
ROLE_CONFIG = {
    "bat": {
        "sample_col": "bat_innings",
        "min_sample": {"SMA": 15, "IPL": 10},
        "impact_weights": {
            "strike_rate": 0.40,
            "runs_per_innings": 0.40,
            "boundary_pct": 0.20,
        },
        "cluster_features": [
            "strike_rate",
            "runs_per_innings",
            "boundary_pct",
            "bat_pp_sr",
            "bat_mid_sr",
            "bat_death_sr",
        ],
    },
    "bowl": {
        "sample_col": "bowl_innings",
        "min_sample": {"SMA": 15, "IPL": 10},
        "impact_weights": {
            "economy": -0.40,
            "wickets_per_innings": 0.35,
            "bowling_sr": -0.25,
        },
        "cluster_features": [
            "economy",
            "bowling_sr",
            "wickets_per_innings",
            "bowl_pp_economy",
            "bowl_mid_economy",
            "bowl_death_economy",
            "wide_rate",
        ],
    },
}


def safe_divide(numerator, denominator, multiplier=1.0):
    return numerator / denominator * multiplier if pd.notna(denominator) and denominator > 0 else np.nan


def aggregate_player_profiles(features, role, competition):
    rows = []
    df = features[(features["role"] == role) & (features["competition"] == competition)].copy()

    for player_key, group in df.groupby("player_key", dropna=False):
        player_name = group["player_name"].dropna().iloc[-1] if group["player_name"].notna().any() else player_key
        player_id = group["player_id"].dropna().iloc[-1] if group["player_id"].notna().any() else player_key
        row = {
            "player_key": player_key,
            "player_id": player_id,
            "player_name": player_name,
            "competition": competition,
            "role": role,
            "seasons": int(group["season"].nunique()),
            "first_season": int(group["season"].min()),
            "last_season": int(group["season"].max()),
        }

        if role == "bat":
            count_cols = [
                "bat_innings", "runs", "balls", "fours", "sixes",
                "bat_pp_runs", "bat_pp_balls", "bat_mid_runs", "bat_mid_balls",
                "bat_death_runs", "bat_death_balls",
            ]
            for col in count_cols:
                row[col] = float(group[col].sum(skipna=True)) if col in group else np.nan
            row["sample"] = row["bat_innings"]
            row["strike_rate"] = safe_divide(row["runs"], row["balls"], 100)
            row["runs_per_innings"] = safe_divide(row["runs"], row["bat_innings"])
            row["boundary_pct"] = safe_divide(row["fours"] + row["sixes"], row["balls"], 100)
            row["bat_pp_sr"] = safe_divide(row["bat_pp_runs"], row["bat_pp_balls"], 100)
            row["bat_mid_sr"] = safe_divide(row["bat_mid_runs"], row["bat_mid_balls"], 100)
            row["bat_death_sr"] = safe_divide(row["bat_death_runs"], row["bat_death_balls"], 100)
        else:
            count_cols = [
                "bowl_innings", "runs_conceded", "balls_bowled", "wickets",
                "bowl_pp_runs", "bowl_pp_balls", "bowl_mid_runs", "bowl_mid_balls",
                "bowl_death_runs", "bowl_death_balls", "wides", "noballs",
            ]
            for col in count_cols:
                row[col] = float(group[col].sum(skipna=True)) if col in group else np.nan
            row["sample"] = row["bowl_innings"]
            row["economy"] = safe_divide(row["runs_conceded"], row["balls_bowled"], 6)
            row["bowling_sr"] = safe_divide(row["balls_bowled"], row["wickets"])
            row["wickets_per_innings"] = safe_divide(row["wickets"], row["bowl_innings"])
            row["wide_rate"] = safe_divide(row["wides"], row["balls_bowled"], 6)
            row["noball_rate"] = safe_divide(row["noballs"], row["balls_bowled"], 6)
            row["bowl_pp_economy"] = safe_divide(row["bowl_pp_runs"], row["bowl_pp_balls"], 6)
            row["bowl_mid_economy"] = safe_divide(row["bowl_mid_runs"], row["bowl_mid_balls"], 6)
            row["bowl_death_economy"] = safe_divide(row["bowl_death_runs"], row["bowl_death_balls"], 6)

        rows.append(row)

    return pd.DataFrame(rows)

profiles = pd.concat(
    [aggregate_player_profiles(features, role, comp) for role in ROLE_CONFIG for comp in ["SMA", "IPL"]],
    ignore_index=True,
    sort=False,
)
profiles["min_sample"] = profiles.apply(lambda r: ROLE_CONFIG[r["role"]]["min_sample"][r["competition"]], axis=1)
eligible_profiles = profiles[profiles["sample"].fillna(0) >= profiles["min_sample"]].copy()

print(f"All profiles: {len(profiles):,}")
print(f"Eligible profiles after sample filters: {len(eligible_profiles):,}")
print(eligible_profiles.groupby(["competition", "role"]).agg(players=("player_key", "nunique"), mean_sample=("sample", "mean")).round(2))


## 3. Observed Impact Scores

The impact score is an observed scouting score, not a future prediction.

- Batter score rewards strike rate, runs per innings, and boundary percentage.
- Bowler score rewards low economy, wicket-taking, and low bowling strike rate.
- Scores are z-scored within each competition and role, then shrunk toward the role average for lower samples.


In [ ]:
def zscore(series):
    std = series.std(ddof=0)
    if pd.isna(std) or std == 0:
        return pd.Series(0.0, index=series.index)
    return (series - series.mean()) / std


def add_observed_impact_scores(profiles):
    out = profiles.copy()
    out["impact_raw"] = np.nan
    out["impact_reliability"] = np.nan
    out["impact_score"] = np.nan
    out["impact_percentile"] = np.nan

    for (competition, role), group in out.groupby(["competition", "role"]):
        weights = ROLE_CONFIG[role]["impact_weights"]
        idx = group.index
        score = pd.Series(0.0, index=idx)
        used_weight = 0.0

        for metric, weight in weights.items():
            values = out.loc[idx, metric]
            if values.notna().sum() < 2:
                continue
            score = score.add(zscore(values).fillna(0.0) * weight, fill_value=0.0)
            used_weight += abs(weight)

        if used_weight == 0:
            continue

        raw = score / used_weight
        reliability = out.loc[idx, "sample"].fillna(0).astype(float) / (out.loc[idx, "sample"].fillna(0).astype(float) + 10.0)
        role_mean = raw.mean()
        impact = reliability * raw + (1.0 - reliability) * role_mean

        out.loc[idx, "impact_raw"] = raw
        out.loc[idx, "impact_reliability"] = reliability
        out.loc[idx, "impact_score"] = impact
        out.loc[idx, "impact_percentile"] = impact.rank(pct=True) * 100.0

    return out

eligible_profiles = add_observed_impact_scores(eligible_profiles)

print("Top observed SMA batters:")
print(eligible_profiles[(eligible_profiles["competition"] == "SMA") & (eligible_profiles["role"] == "bat")]
      .sort_values("impact_score", ascending=False)
      [["player_name", "sample", "impact_score", "impact_percentile", "strike_rate", "runs_per_innings", "boundary_pct"]]
      .head(10)
      .round(2)
      .to_string(index=False))

print("\nTop observed SMA bowlers:")
print(eligible_profiles[(eligible_profiles["competition"] == "SMA") & (eligible_profiles["role"] == "bowl")]
      .sort_values("impact_score", ascending=False)
      [["player_name", "sample", "impact_score", "impact_percentile", "economy", "bowling_sr", "wickets_per_innings"]]
      .head(10)
      .round(2)
      .to_string(index=False))


## 4. K-means Player Profiles

K-means is used to group similar players by role. This answers the scouting question:

> What type of player is this?

The clusters are not treated as right/wrong labels. They are scouting profiles that help compare players with similar roles.


In [ ]:
def label_cluster(role, row, role_summary):
    if role == "bat":
        if row["impact_score"] == role_summary["impact_score"].max():
            return "High-impact batting profile"
        if row["strike_rate"] >= role_summary["strike_rate"].quantile(0.75) and row["boundary_pct"] >= role_summary["boundary_pct"].median():
            return "Power-hitting profile"
        if row["runs_per_innings"] == role_summary["runs_per_innings"].max():
            return "Run-accumulation profile"
        if row.get("bat_death_sr", np.nan) == role_summary["bat_death_sr"].max():
            return "Finishing profile"
        return "Balanced batting profile"

    if row["impact_score"] == role_summary["impact_score"].max():
        return "High-impact bowling profile"
    if row["economy"] == role_summary["economy"].min():
        return "Economy-control profile"
    if row["wickets_per_innings"] == role_summary["wickets_per_innings"].max():
        return "Wicket-taking profile"
    if row["bowling_sr"] == role_summary["bowling_sr"].min():
        return "Strike-bowling profile"
    return "Balanced bowling profile"

cluster_outputs = []
role_models = {}
eligible_profiles["cluster"] = np.nan
eligible_profiles["profile_type"] = pd.NA

for role, cfg in ROLE_CONFIG.items():
    role_idx = eligible_profiles[eligible_profiles["role"] == role].index
    role_df = eligible_profiles.loc[role_idx].copy()
    feature_cols = [col for col in cfg["cluster_features"] if col in role_df.columns]

    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    x_imputed = imputer.fit_transform(role_df[feature_cols].replace([np.inf, -np.inf], np.nan))
    x_scaled = scaler.fit_transform(x_imputed)

    n_clusters = min(4, len(role_df))
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=20)
    labels = kmeans.fit_predict(x_scaled)

    eligible_profiles.loc[role_idx, "cluster"] = labels
    role_models[role] = {"features": feature_cols, "imputer": imputer, "scaler": scaler, "kmeans": kmeans}

    summary = role_df.assign(cluster=labels).groupby("cluster").agg(
        players=("player_key", "nunique"),
        sma_players=("competition", lambda s: int((s == "SMA").sum())),
        ipl_players=("competition", lambda s: int((s == "IPL").sum())),
        mean_sample=("sample", "mean"),
        impact_score=("impact_score", "mean"),
        **{col: (col, "mean") for col in feature_cols},
    ).reset_index()
    summary.insert(0, "role", role)
    summary["profile_type"] = summary.apply(lambda r: label_cluster(role, r, summary), axis=1)
    cluster_outputs.append(summary)

    label_map = dict(zip(summary["cluster"], summary["profile_type"]))
    eligible_profiles.loc[role_idx, "profile_type"] = eligible_profiles.loc[role_idx, "cluster"].map(label_map)

cluster_summary = pd.concat(cluster_outputs, ignore_index=True, sort=False)
eligible_profiles["cluster"] = eligible_profiles["cluster"].astype(int)

print(cluster_summary.round(2).to_string(index=False))


## 5. Similar IPL Player Matching

For each eligible domestic SMA player who has not appeared in IPL data, the notebook finds the closest IPL players in the same role using standardized profile features.

This answers the scouting question:

> Which IPL players does this domestic player resemble?


In [ ]:
def build_similarity_matches(profiles, top_n=3):
    rows = []
    for role, cfg in ROLE_CONFIG.items():
        model = role_models[role]
        feature_cols = model["features"]

        role_profiles = profiles[profiles["role"] == role].copy()
        ipl_keys = set(role_profiles[role_profiles["competition"] == "IPL"]["player_key"])
        candidates = role_profiles[(role_profiles["competition"] == "SMA") & (~role_profiles["player_key"].isin(ipl_keys))].reset_index(drop=True)
        comparables = role_profiles[role_profiles["competition"] == "IPL"].reset_index(drop=True)

        if candidates.empty or comparables.empty:
            continue

        comparable_x = model["scaler"].transform(model["imputer"].transform(comparables[feature_cols].replace([np.inf, -np.inf], np.nan)))
        candidate_x = model["scaler"].transform(model["imputer"].transform(candidates[feature_cols].replace([np.inf, -np.inf], np.nan)))

        nearest = NearestNeighbors(n_neighbors=min(top_n, len(comparables)), metric="euclidean")
        nearest.fit(comparable_x)
        distances, indices = nearest.kneighbors(candidate_x)

        for candidate_pos, candidate in candidates.iterrows():
            for match_rank, (distance, comparable_pos) in enumerate(zip(distances[candidate_pos], indices[candidate_pos]), start=1):
                comparable = comparables.iloc[int(comparable_pos)]
                rows.append({
                    "domestic_player_key": candidate["player_key"],
                    "domestic_player_name": candidate["player_name"],
                    "role": role,
                    "domestic_sample": candidate["sample"],
                    "domestic_impact_score": candidate["impact_score"],
                    "domestic_impact_percentile": candidate["impact_percentile"],
                    "domestic_profile_type": candidate["profile_type"],
                    "match_rank": match_rank,
                    "ipl_player_key": comparable["player_key"],
                    "ipl_player_name": comparable["player_name"],
                    "ipl_sample": comparable["sample"],
                    "ipl_impact_score": comparable["impact_score"],
                    "ipl_impact_percentile": comparable["impact_percentile"],
                    "ipl_profile_type": comparable["profile_type"],
                    "distance": float(distance),
                    "similarity_score": float(100.0 / (1.0 + distance)),
                })

    return pd.DataFrame(rows)

similarity_matches = build_similarity_matches(eligible_profiles, top_n=3)
print(f"Similarity match rows: {len(similarity_matches):,}")
print(similarity_matches.head(12).round(2).to_string(index=False))


## 6. Final Scouting Shortlist

The final shortlist combines three scouting signals:

- 60% observed SMA impact percentile.
- 25% similarity strength to proven IPL profiles.
- 15% sample-size percentile.

This is intentionally a scouting ranking, not a prediction of future IPL performance.


In [ ]:
def add_role_percentile(df, col, out_col):
    df[out_col] = df.groupby("role")[col].rank(pct=True) * 100.0
    return df

rank1_matches = similarity_matches[similarity_matches["match_rank"] == 1].copy()
rank1_matches = rank1_matches.rename(columns={
    "ipl_player_name": "closest_ipl_player",
    "ipl_profile_type": "closest_ipl_profile_type",
    "ipl_impact_percentile": "closest_ipl_impact_percentile",
})

candidate_profiles = eligible_profiles[
    (eligible_profiles["competition"] == "SMA")
    & (~eligible_profiles["player_key"].isin(set(eligible_profiles[eligible_profiles["competition"] == "IPL"]["player_key"])))
].copy()

shortlist = candidate_profiles.merge(
    rank1_matches[[
        "domestic_player_key", "role", "closest_ipl_player", "closest_ipl_profile_type",
        "closest_ipl_impact_percentile", "similarity_score", "distance",
    ]],
    left_on=["player_key", "role"],
    right_on=["domestic_player_key", "role"],
    how="left",
)
shortlist = add_role_percentile(shortlist, "sample", "sample_percentile")
shortlist = add_role_percentile(shortlist, "similarity_score", "similarity_percentile")
shortlist["scouting_score"] = (
    0.60 * shortlist["impact_percentile"].fillna(0)
    + 0.25 * shortlist["similarity_percentile"].fillna(0)
    + 0.15 * shortlist["sample_percentile"].fillna(0)
)


def scouting_reason(row):
    if row["role"] == "bat":
        return (
            f"{row['profile_type']}; SR {row['strike_rate']:.1f}, "
            f"RPI {row['runs_per_innings']:.1f}, boundary% {row['boundary_pct']:.1f}; "
            f"closest IPL comp: {row['closest_ipl_player']}"
        )
    return (
        f"{row['profile_type']}; economy {row['economy']:.2f}, "
        f"bowling SR {row['bowling_sr']:.1f}, WPI {row['wickets_per_innings']:.2f}; "
        f"closest IPL comp: {row['closest_ipl_player']}"
    )

shortlist["scouting_reason"] = shortlist.apply(scouting_reason, axis=1)

output_cols = [
    "player_name", "role", "sample", "scouting_score", "impact_score", "impact_percentile",
    "profile_type", "closest_ipl_player", "closest_ipl_profile_type", "similarity_score",
    "strike_rate", "runs_per_innings", "boundary_pct",
    "economy", "bowling_sr", "wickets_per_innings",
    "scouting_reason",
]
output_cols = [col for col in output_cols if col in shortlist.columns]

scouting_shortlist = shortlist.sort_values(["role", "scouting_score"], ascending=[True, False]).groupby("role").head(25).reset_index(drop=True)

print("Top domestic SMA batters:")
print(scouting_shortlist[scouting_shortlist["role"] == "bat"][output_cols].head(10).round(2).to_string(index=False))

print("\nTop domestic SMA bowlers:")
print(scouting_shortlist[scouting_shortlist["role"] == "bowl"][output_cols].head(10).round(2).to_string(index=False))


## 7. Save Scouting Results

These CSV files are the outputs that can feed the dashboard and final written report.


In [ ]:
eligible_profiles.to_csv(SCOUTING_DIR / "player_profiles_with_clusters.csv", index=False)
cluster_summary.to_csv(SCOUTING_DIR / "cluster_summary.csv", index=False)
similarity_matches.to_csv(SCOUTING_DIR / "player_similarity_matches.csv", index=False)
shortlist.sort_values(["role", "scouting_score"], ascending=[True, False]).to_csv(SCOUTING_DIR / "domestic_candidates_scored.csv", index=False)
scouting_shortlist.to_csv(SCOUTING_DIR / "scouting_shortlist.csv", index=False)

print("Saved scouting outputs:")
for path in sorted(SCOUTING_DIR.glob("*.csv")):
    print(f"- {path}")


## 8. Quick Visual Checks

The charts below are optional dashboard/report figures. The CSV outputs are the most important artifacts.


In [ ]:
def save_top_chart(df, role, metric, title, filename):
    plot_df = df[df["role"] == role].sort_values(metric, ascending=True).tail(15)
    plt.figure(figsize=(9, 6))
    plt.barh(plot_df["player_name"], plot_df[metric], color="#2f6f73" if role == "bat" else "#c57b57")
    plt.title(title)
    plt.xlabel(metric.replace("_", " ").title())
    plt.tight_layout()
    plt.savefig(SCOUTING_DIR / filename, dpi=160)
    plt.close()

save_top_chart(scouting_shortlist, "bat", "scouting_score", "Top Domestic SMA Batters", "top_domestic_batters.png")
save_top_chart(scouting_shortlist, "bowl", "scouting_score", "Top Domestic SMA Bowlers", "top_domestic_bowlers.png")

bat_plot = eligible_profiles[eligible_profiles["role"] == "bat"].dropna(subset=["strike_rate", "runs_per_innings"])
plt.figure(figsize=(8, 6))
for label, group in bat_plot.groupby("profile_type"):
    plt.scatter(group["strike_rate"], group["runs_per_innings"], s=30, alpha=0.65, label=label)
plt.title("Batting Profiles: Strike Rate vs Runs Per Innings")
plt.xlabel("Strike Rate")
plt.ylabel("Runs Per Innings")
plt.legend(fontsize=7)
plt.tight_layout()
plt.savefig(SCOUTING_DIR / "batting_profile_clusters.png", dpi=160)
plt.close()

bowl_plot = eligible_profiles[eligible_profiles["role"] == "bowl"].dropna(subset=["economy", "wickets_per_innings"])
plt.figure(figsize=(8, 6))
for label, group in bowl_plot.groupby("profile_type"):
    plt.scatter(group["economy"], group["wickets_per_innings"], s=30, alpha=0.65, label=label)
plt.title("Bowling Profiles: Economy vs Wickets Per Innings")
plt.xlabel("Economy")
plt.ylabel("Wickets Per Innings")
plt.legend(fontsize=7)
plt.tight_layout()
plt.savefig(SCOUTING_DIR / "bowling_profile_clusters.png", dpi=160)
plt.close()

print("Saved scouting charts:")
for path in sorted(SCOUTING_DIR.glob("*.png")):
    print(f"- {path}")


## Final Interpretation

This notebook creates a practical scouting workflow:

- The impact score ranks observed domestic and IPL performance.
- K-means clusters explain the type of player.
- Nearest-neighbor matching gives IPL comparables.
- The final shortlist identifies domestic SMA players worth scouting more closely.

This framing avoids overclaiming that SMA data can accurately predict future IPL success. Instead, it gives scouts a structured way to compare, filter, and investigate players.
